# ollama


In [ ]:
!pwd


## amd gpu


In [ ]:
!sudo pacman -S --needed --noconfirm rocm-opencl-runtime rocm-hip-runtime


## install


In [ ]:
%%bash
path_append() {
	local tgt=${1:-}
	[[ -z "$tgt" ]] && return 1

	local e_l="export PATH=\"$tgt:\$PATH\""
	if grep -qF "$e_l" "$HOME/.zshrc"; then
		echo "  ✓ PATH already contains $tgt"
	else
		echo "$e_l" >>"$HOME/.zshrc"
		echo "  ✓ Added $tgt to PATH"
	fi
}
install_ollama(){
    local install_dir=${1:-/data/.path/.ollama}
    [[ -d "$install_dir" ]] && return
    cd ~/Downloads
    # 检查文件是否存在，不存在才下载
    # aria2 专用代理参数
    proxy_opts="--all-proxy=https://192.168.0.103:7897"
    
    # 使用 aria2 下载，避免重复下载
    [[ ! -f "ollama-linux-amd64.tgz" ]] && \
        aria2c -x 16 -s 16 --continue=true $proxy_opts \
        https://ollama.com/download/ollama-linux-amd64.tgz
    
    [[ ! -f "ollama-linux-amd64-rocm.tgz" ]] && \
        aria2c -x 16 -s 16 --continue=true $proxy_opts \
        https://ollama.com/download/ollama-linux-amd64-rocm.tgz
    # 解压文件
    mkdir -p "$install_dir"
    tar -zxf ollama-linux-amd64.tgz -C "$install_dir"
    tar -zxf ollama-linux-amd64-rocm.tgz -C "$install_dir"

    # 检查组是否存在，不存在才创建
    getent group ollama >/dev/null || sudo groupadd ollama
    groups $USER | grep -q ollama || sudo usermod -aG ollama $USER
    
    rm -rf $HOME/.ollama
    ln -sf "$install_dir" "$HOME/.ollama"
    # 添加环境变量
    path_append "$install_dir/bin" 
    
    echo "  ✓ ollama installed"
    echo "source $HOME/.zshrc"
    
}

install_ollama


## start serve


In [ ]:
%%bash
## run
HSA_OVERRIDE_GFX_VERSION=10.3.0 ollama serve


## run model


In [ ]:
%%bash
ollama list
ollama run ollama3.2:3b


## python 调用


In [ ]:
conda create -n ollama python ipykernel -y


In [ ]:
%pip install ollama


In [ ]:
import ollama

# 测试模型
response = ollama.generate(
    model='deepseek-r1',
    prompt='解释一下什么是趋势交易'
)
print(f"{response['response']}")


In [ ]:
# 处理所有 SRT 文件
from pathlib import Path
def extract_srt_text(srt_file):
    with open(srt_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    text_lines = []
    for line in content.split('\n'):
        line = line.strip()
        if line and not line.isdigit() and '-->' not in line:
            text_lines.append(line)
    
    return ' '.join(text_lines)
folder = Path("/data/projects/TikTokDownloader/Volume/UID72889236818_天启大烁哥_发布作品")
srt_files = sorted(folder.glob("*.srt"))


In [ ]:
# 合并内容
all_content = []
for srt_file in srt_files:
    text = extract_srt_text(srt_file)
    if text.strip():
        title = srt_file.stem.split('-视频-天启大烁哥-')[-1].split('#')[0]
        all_content.append(f"【{title}】\n{text}")

combined_text = "\n\n".join(all_content)
print(f"总字符数: {len(combined_text):,}")


In [ ]:
# 投喂大模型 - 详细版
prompt = f"""你是一位资深的期货交易专家和教育者。请深入分析以下天启大烁哥的122个期货交易视频内容（共21万字），提供一份详尽的交易理念总结报告。

{combined_text}

"""

response = ollama.generate(
    model='deepseek-r1',
    prompt=prompt,
    options={
        'temperature': 0.7,
        'num_predict': 8000  # 增加输出长度
    }
)

print(response['response'])

# 保存结果
with open(f"{folder}/deepseek_详细总结.md", 'w', encoding='utf-8') as f:
    f.write("# 天启大烁哥期货交易理念详细总结\n\n")
    f.write(response['response'])

print(f"\n详细总结已保存到: {folder}/deepseek_详细总结.md")
